In [44]:
import pandas as pd
import numpy as np
import h5py
import os

In [45]:
root = "."
df = pd.read_csv(os.path.join(root, "..", "processed", "enamine_REAL_characterization", "enamine_REAL.tsv"), sep='\t')
df['index'] = df.index

In [46]:
smiles = df['smiles'].tolist()

In [47]:
with open("/home/acomajuncosa/Documents/playground/fp2sim/smiles.smi", "w") as f:
    f.write("\n".join([i.split()[0] + " " + str(c) for c, i in enumerate(smiles)]))

In [5]:
from FPSim2 import FPSim2Engine

In [57]:
fp_filename = "/home/acomajuncosa/Documents/playground/fp2sim/fp_db.h5"
fpe = FPSim2Engine(fp_filename)

query = 'COC1=CC=CC(NNC(=O)C2=CC=CC(I)=C2O)=C1Cl'
results = fpe.similarity(query, threshold=-1, metric='tanimoto', n_workers=16)

In [59]:
results

array([(      0, 1.        ), (8901134, 0.56363636),
       (5658675, 0.54385966), ..., (4770713, 0.01030928),
       (6026737, 0.        ), (5494694, 0.        )],
      shape=(9557694,), dtype={'names': ['mol_id', 'coeff'], 'formats': ['<u4', '<f4'], 'offsets': [4, 8], 'itemsize': 12})

In [33]:
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs

def tanimoto(smiles_a: str, smiles_b: str, radius=2, n_bits=2048) -> float:
    m1, m2 = Chem.MolFromSmiles(smiles_a), Chem.MolFromSmiles(smiles_b)
    if m1 is None or m2 is None:
        raise ValueError("One or both SMILES are invalid.")
    fp1 = AllChem.GetMorganFingerprintAsBitVect(m1, radius, nBits=n_bits)
    fp2 = AllChem.GetMorganFingerprintAsBitVect(m2, radius, nBits=n_bits)
    return DataStructs.TanimotoSimilarity(fp1, fp2)